In [ ]:
# Notebook 3 — Transformer tự build (train from scratch) cho bài toán
# khôi phục dấu tiếng Việt, + so sánh 4 hệ + phân tích lỗi + demo Gradio.
# Notebook này độc lập với Notebook 1 nhưng tải lại đúng corpus 100k câu
# (cùng code, cùng seed 42) nên train/val/test giống hệt Notebook 1.
# Công thức bài toán: phân loại token ở mức TỪ — mỗi từ không dấu được
# xếp vào 1 MẪU DẤU (pattern): các ký tự có dấu tại những vị trí có thể
# mang dấu của từ đó. Vd "được" -> mẫu "đượ", "có" -> "ó", "hoàn" -> "oà".
# Cùng một mẫu có nghĩa giống nhau cho mọi từ nên model học được quy
# luật dấu từ ngữ cảnh (xem Cell 2). Model: Transformer encoder nhỏ
# (4 lớp, hidden 256, 8 heads, ~9,5M tham số), train 12 epoch.
# Cập nhật 17/11 (review code): thêm torch.manual_seed (Cell 3), assert
# kiểm tra khớp dữ liệu (Cell 1), masking mẫu tương thích khi dự đoán
# (Cell 5/9). Số trong báo cáo (92,5%) đo ngày 16/11, trước các sửa đổi
# này — chạy lại có thể lệch nhẹ (seed làm train tái tạo được; masking
# chỉ có thể làm số tăng lên, không giảm).

# Cell 1 — Hàm bỏ dấu + tải Wikipedia + chia train/val/test
import unicodedata, re, random, collections

def strip_diacritics(s: str) -> str:
    # Bỏ dấu tiếng Việt: tách dấu bằng NFD rồi loại bỏ ký tự dấu,
    # riêng đ/Đ xử lý tay vì NFD không tách được.
    s = s.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if unicodedata.category(c) != "Mn")

from datasets import load_dataset

N_ARTICLES = 10000
TARGET_SENTENCES = 100000

ds = load_dataset("wikimedia/wikipedia", "20231101.vi",
                  split="train", streaming=True)
sentences, seen = [], set()
n_read = 0
for ex in ds:
    n_read += 1
    if n_read > N_ARTICLES:
        break
    text = ex["text"].replace("\n", " ")
    for s in re.split(r"(?<=[.!?…])\s+(?=[A-ZÀ-ỸĐ])", text):
        s = re.sub(r"\s+", " ", s).strip()
        w = s.split()
        if not (5 <= len(w) <= 40):
            continue
        if "http" in s or s.count("=") > 2:
            continue
        if sum(c.isdigit() for c in s) / max(len(s), 1) > 0.15:
            continue
        if s in seen:
            continue
        seen.add(s)
        sentences.append(s)
        if len(sentences) >= TARGET_SENTENCES:
            break
    if len(sentences) >= TARGET_SENTENCES:
        break
print(f"bai viet da doc: {n_read} | cau giu lai: {len(sentences)}")

random.seed(42)
# Chuẩn hóa câu về NFC: Wikipedia có chỗ lưu dấu dạng tách rời (NFD,
# vd "ê" = "e" + dấu mũ tách riêng) khiến từ có dấu và từ không dấu
# lệch độ dài. NFC gộp mỗi ký tự có dấu thành 1 ký tự duy nhất.
pairs = [(strip_diacritics(s), unicodedata.normalize("NFC", s))
         for s in sentences]
random.shuffle(pairs)
n = len(pairs)
n_train, n_val = int(n * 0.8), int(n * 0.1)
train = pairs[:n_train]
val = pairs[n_train:n_train + n_val]
test = pairs[n_train + n_val:]
print(f"train {len(train)} | val {len(val)} | test {len(test)}")

# Kiểm tra khớp với Notebook 1: tổng số từ phải = 2.387.449 —
# nếu khác thì tập dữ liệu đã lệch.
words = [w for _, acc in pairs for w in acc.split()]
print(f"tong so tu: {len(words)}")
# Dừng ngay nếu tập dữ liệu lệch khỏi Notebook 1.
assert len(words) == 2387449, f"tap du lieu lech: {len(words)} tu (muon 2.387.449)"

In [ ]:
# Cell 2 — Mã hóa dấu thành "mẫu dấu" (signature)
# Thay vì bắt model chọn 1 trong các dạng có dấu của từng từ (nhãn khác
# nhau cho mỗi từ, model khó học), ta tách riêng PHẦN DẤU ra khỏi từ.
# Mẫu dấu của một từ = tuple các đơn vị có dấu tại những vị trí có thể
# mang dấu, theo đúng thứ tự. Ví dụ: "được" -> ("đ", "ư", "ợ");
# "có" -> ("ó",); "hoàn" -> ("o", "à") (o giữ nguyên, a->à).
# Từ không có vị trí mang dấu (vd "123", "Mr") có mẫu rỗng ().
# MODIFIABLE để CỐ ĐỊNH theo tiếng Việt (nguyên âm a e i o u y + đ),
# không học từ dữ liệu: Wikipedia chứa chữ nước ngoài (Thái, Khmer,
# Hy Lạp...) nên học từ dữ liệu làm số mẫu dấu bùng nổ tới hàng chục
# nghìn và model học kém. Tên riêng nước ngoài chứa dấu phụ âm ngoài
# tiếng Việt (vd "Rêš", "François") thì không mã hóa được bằng mẫu
# dấu tiếng Việt — chúng chiếm rất ít (0,03%) và được bỏ qua khi train.
# Mẫu dấu lưu dạng tuple vì Wikipedia có chỗ lưu dấu dạng tách rời
# (vd "ê" = "e" + dấu mũ riêng): tách từ thành đơn vị (ký tự gốc +
# dấu đi kèm) bằng NFD rồi gộp mỗi đơn vị về NFC thì mã hóa đúng
# với mọi dạng lưu trữ.
from collections import Counter, defaultdict

# Vị trí có thể mang dấu trong tiếng Việt (cố định, không học từ data)
MODIFIABLE = set("aeiouyAEIOUYdD")

def split_units(word):
    # Tách từ thành các đơn vị: mỗi đơn vị = 1 ký tự gốc + dấu đi kèm.
    # Vd "Rês" -> ["R", "ê"] dù "ê" lưu dạng gộp hay tách rời.
    units = []
    for ch in unicodedata.normalize("NFD", word):
        if unicodedata.category(ch) == "Mn" and units:
            units[-1] += ch  # dấu đi kèm ký tự trước nó
        else:
            units.append(ch)
    # Gộp mỗi đơn vị về NFC cho gọn (vd "e" + dấu mũ -> "ê")
    return [unicodedata.normalize("NFC", u) for u in units]

def apply_signature(bare, sig):
    # Khôi phục từ có dấu từ từ không dấu + mẫu dấu.
    # Mẫu phải có đúng số đơn vị = số vị trí có thể mang dấu.
    chars = list(bare)
    pos = [i for i, b in enumerate(bare) if b in MODIFIABLE]
    if len(sig) != len(pos):
        return None  # mẫu không khớp từ này
    for i, s in zip(pos, sig):
        chars[i] = s
    return "".join(chars)

def to_signature(bare, acc):
    # Mẫu dấu = tuple các đơn vị có dấu tại vị trí có thể mang dấu.
    # Trả về None nếu từ không mã hóa được bằng mẫu dấu tiếng Việt:
    # số đơn vị khác số ký tự gốc, hoặc khôi phục lại không khớp gold
    # (vd tên riêng nước ngoài chứa dấu phụ âm ngoài tiếng Việt).
    units = split_units(acc)
    if len(units) != len(bare):
        return None
    sig = tuple(u for u, b in zip(units, bare) if b in MODIFIABLE)
    got = apply_signature(bare, sig)
    if got is None or unicodedata.normalize("NFC", got) != unicodedata.normalize("NFC", acc):
        return None
    return sig

# Chẩn đoán: đếm từ chưa ở dạng NFC (Cell 1 đã chuẩn hóa NFC nên
# con số này phải = 0; nếu > 0 nghĩa là bước chuẩn hóa chưa chạy).
n_non_nfc = sum(1 for _, acc in train for a in acc.split()
                if unicodedata.normalize("NFC", a) != a)
print(f"so tu chua o dang NFC trong train: {n_non_nfc}")

# Thống kê mã hóa trên toàn bộ train: từ không mã hóa được (chủ yếu
# tên riêng nước ngoài) sẽ mang nhãn -100 khi train và được tính là
# lỗi khi đánh giá — tỉ lệ của chúng phải rất nhỏ.
n_total = n_unencodable = 0
unenc_examples = []
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        n_total += 1
        if to_signature(b, a) is None:
            n_unencodable += 1
            if len(unenc_examples) < 5:
                unenc_examples.append((b, a))
print(f"tong so tu train: {n_total} | khong ma hoa duoc: "
      f"{n_unencodable} ({n_unencodable / n_total:.3%})")
for b, a in unenc_examples:
    print(f"  vd khong ma hoa duoc: {b!r} -> {a!r}")
assert n_unencodable / n_total < 0.01, "ti le tu khong ma hoa duoc qua cao"

sig_counts = Counter()
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        sig = to_signature(b, a)
        if sig is not None:
            sig_counts[sig] += 1
# Tập nhãn = các mẫu dấu xuất hiện từ 5 lần trở lên trong train.
# Phần lớn mẫu chỉ xuất hiện 1-2 lần (đuôi dài của từ hiếm) — giữ
# chúng thì số lớp bùng lên hàng nghìn mà model không đủ ví dụ để
# học; bỏ chúng đi thì các từ đó mang nhãn -100 khi train (không
# đóng góp vào loss), khi dự đoán model chọn mẫu tương thích phổ
# biến nhất. Không có mẫu nào bị mất khỏi dữ liệu, chỉ không làm nhãn.
MIN_SIG_COUNT = 5
signatures = sorted(s for s, c in sig_counts.items() if c >= MIN_SIG_COUNT)
sig_to_id = {s: i for i, s in enumerate(signatures)}
NUM_LABELS = len(signatures)
n_dropped = sum(1 for c in sig_counts.values() if c < MIN_SIG_COUNT)
n_dropped_tok = sum(c for c in sig_counts.values() if c < MIN_SIG_COUNT)
print(f"so mau dau khac nhau (giu lai): {NUM_LABELS}")
print(f"so mau bi loai (xuat hien < {MIN_SIG_COUNT} lan): {n_dropped} | "
      f"so tu bi anh huong: {n_dropped_tok}")
print("10 mau pho bien nhat:",
      [("".join(s), c) for s, c in sig_counts.most_common(10)])

# Với mỗi ký tự gốc, tập các đơn vị có dấu đã thấy (để khóa các mẫu
# không tương thích khi dự đoán, vd mẫu ("ó",) không dùng được cho "ba").
valid_forms = defaultdict(set)
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        units = split_units(a)
        if len(units) != len(b):
            continue
        for bc, unit in zip(b, units):
            if bc in MODIFIABLE:
                valid_forms[bc].add(unit)

# Tra cứu mẫu tương thích cho một từ (dùng khi dự đoán ở Cell 5/9):
# chỉ xét các mẫu có đúng độ dài và từng ký tự là dạng có dấu hợp lệ
# của ký tự gốc — model không được chọn mẫu "vô lý" cho từ đó
# (vd mẫu ("ó",) không dùng được cho từ "ba" vì "b" không mang dấu).
def skeleton(bare):
    return tuple(b for b in bare if b in MODIFIABLE)

_by_len = defaultdict(list)
for _i, _s in enumerate(signatures):
    _by_len[len(_s)].append(_i)

_compat_cache = {}
def compatible_ids(skel):
    # Danh sách id các mẫu dấu tương thích với một bộ khung.
    if skel not in _compat_cache:
        _compat_cache[skel] = [i for i in _by_len[len(skel)] if all(
            signatures[i][j] in valid_forms[skel[j]]
            for j in range(len(skel)))]
    return _compat_cache[skel]

# Tập từ đa nghĩa (dùng để đo accuracy riêng, giống Notebook 1)
forms = defaultdict(set)
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        forms[b].add(a)
amb_set = {b for b, v in forms.items() if len(v) >= 2}
print(f"tu da nghia: {len(amb_set)}")

In [ ]:
# Cell 3 — Từ điển từ + Dataset + model Transformer encoder nhỏ
# Model tự build làm việc ở mức TỪ (không dùng subword như PhoBERT):
# mỗi vị trí = 1 từ, nhãn = mẫu dấu của từ đó.
# Từ điển: giữ từ không dấu xuất hiện >= 2 lần trong train, thêm
# <pad>=0, <unk>=1. Từ chưa thấy ở test -> <unk>, model đoán mẫu
# theo ngữ cảnh xung quanh.
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "chua bat GPU — vao Settings > Accelerator > GPU T4 x2"

# Seed cho PyTorch (khởi tạo trọng số, dropout, thứ tự xáo trộn của
# DataLoader) để lần chạy lại cho cùng kết quả. random.seed(42) ở
# Cell 1 chỉ seed phần chia dữ liệu, không seed được PyTorch.
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

word_counter = Counter()
for bare, _ in train:
    word_counter.update(bare.split())
vocab = ["<pad>", "<unk>"] + sorted([w for w, c in word_counter.items() if c >= 2])
w2id = {w: i for i, w in enumerate(vocab)}
print(f"vocab tu khong dau: {len(vocab)}")

MAX_LEN = 64  # câu dài nhất 40 từ theo cách lọc ở Cell 1, lấy dư thành 64

def encode_sentence(bare_sent):
    # Mã hóa câu thành danh sách id từ, cắt ở MAX_LEN.
    ids = [w2id.get(w, 1) for w in bare_sent.split()]
    return ids[:MAX_LEN]

class DiacriticDataset(Dataset):
    # Mỗi mẫu: (input_ids, labels) — labels là id mẫu dấu cho từng
    # từ, -100 cho từ không mã hóa được hoặc có mẫu hiếm (< 5 lần).
    def __init__(self, pairs):
        self.items = []
        for bare, acc in pairs:
            b_words, a_words = bare.split(), acc.split()
            labels = []
            for b, a in zip(b_words, a_words):
                sig = to_signature(b, a)
                labels.append(sig_to_id.get(sig, -100)
                              if sig is not None else -100)
            self.items.append((encode_sentence(bare), labels[:MAX_LEN]))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    # Đệm các câu trong batch về cùng độ dài; labels đệm bằng -100.
    maxlen = max(len(ids) for ids, _ in batch)
    xs, ys = [], []
    for ids, labels in batch:
        pad = maxlen - len(ids)
        xs.append(ids + [0] * pad)
        ys.append(labels + [-100] * pad)
    return torch.tensor(xs), torch.tensor(ys)

# Val không cần Dataset/DataLoader riêng: Cell 4 đo word accuracy thật
# trên val bằng cách khôi phục từng câu thô (giống hệt cách đo test).
train_ds = DiacriticDataset(train)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate)
print(f"train {len(train_ds)} | val {len(val)}")

class SmallTransformer(nn.Module):
    def __init__(self, vocab_size, num_labels, d_model=256, nhead=8,
                 num_layers=4, dim_ff=1024, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos = nn.Embedding(MAX_LEN, d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                           dim_feedforward=dim_ff, dropout=dropout,
                                           batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, num_labels)
    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h = self.emb(x) + self.pos(pos)
        pad_mask = (x == 0)
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        return self.head(h)

model = SmallTransformer(len(vocab), NUM_LABELS).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"model: {n_params:,} tham so (encoder 4 lop, hidden 256, 8 heads, dropout 0,1)")

def restore_with_model(mdl, bare_sent):
    # Khôi phục dấu cho 1 câu bằng model cho trước: với mỗi từ, chỉ xét
    # các mẫu dấu TƯƠNG THÍCH với từ đó (compatible_ids, Cell 2), lấy
    # mẫu có logit cao nhất rồi giải mã. Dùng chung cho đánh giá val
    # (Cell 4), đánh giá test (Cell 5) và demo (Cell 9) để 3 chỗ đo
    # cùng một cách.
    ids = torch.tensor([encode_sentence(bare_sent)], device=device)
    with torch.no_grad():
        logits = mdl(ids)[0].cpu()
    out = []
    for b, lg in zip(bare_sent.split(), logits):
        cands = compatible_ids(skeleton(b))
        if not cands:
            out.append(b)
            continue
        best = max(cands, key=lambda k: lg[k].item())
        restored = apply_signature(b, signatures[best])
        out.append(restored if restored is not None else b)
    return " ".join(out)

def restore_sentence(bare_sent):
    # Khôi phục dấu bằng model đang train (Cell 4/5).
    return restore_with_model(model, bare_sent)

In [ ]:
# Cell 4 — Train
# Train from scratch: Adam lr 1e-3, 12 epoch, giữ checkpoint tốt nhất
# theo word accuracy trên val (không lấy epoch cuối một cách mù quáng).
import math, time

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
EPOCHS = 12
best_val_acc, best_state = -1.0, None

def val_word_acc():
    # Word accuracy THẬT trên val: khôi phục từng câu (có masking,
    # giống hệt cách đo test ở Cell 5) rồi so từng từ với gold.
    # Trước 17/11 hàm này đo accuracy trên nhãn mẫu (loại từ -100)
    # nên số val không so sánh được trực tiếp với số test — cách đo
    # khác nhau là nguyên nhân nghi ngờ của khoảng chênh val 91,1%
    # so với test 92,5% ở lần chạy 16/11; đo cùng cách sẽ rõ.
    # Đo từng câu nên mỗi epoch chậm thêm ~30 giây (10k câu val).
    model.eval()
    cor = tot = 0
    for bare, acc in val:
        pred = restore_sentence(bare)
        for p, a in zip(pred.split(), acc.split()):
            tot += 1
            cor += (p == a)
    model.train()
    return cor / tot

for ep in range(1, EPOCHS + 1):
    t0, tot_loss, steps = time.time(), 0.0, 0
    model.train()
    for xs, ys in train_dl:
        xs, ys = xs.to(device), ys.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xs).reshape(-1, NUM_LABELS), ys.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tot_loss += loss.item()
        steps += 1
    va = val_word_acc()
    print(f"epoch {ep}/{EPOCHS} | loss {tot_loss/steps:.4f} | "
          f"val word-acc {va:.1%} | {time.time()-t0:.0f}s")
    if va > best_val_acc:
        best_val_acc = va
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  -> checkpoint tot nhat moi (val {va:.1%})")

model.load_state_dict(best_state)
torch.save(best_state, "/kaggle/working/selfbuilt_best.pt")
print(f"Train xong. Val tot nhat: {best_val_acc:.1%} — da luu selfbuilt_best.pt")

In [ ]:
# Cell 5 — Đo đúng 3 metric như baseline/PhoBERT, trên CÙNG tập test 10k.
# Lưu file dự đoán để chặng 6 tính lại cả 3 metric theo MỘT định nghĩa chung.
import csv

# Tập đa nghĩa dùng cho metric lấy từ Cell 2 (amb_set, tính trên train,
# 5.461 từ) để mẫu số khớp PhoBERT (207.644).
print(f"tu da nghia (tren train): {len(amb_set)}")

# restore_sentence dùng chung định nghĩa ở Cell 3 (có masking mẫu tương
# thích; trước 17/11 cell này lấy argmax trần trên mọi nhãn — số 92,5%
# trong báo cáo đo bằng argmax trần).

model.eval()
tot = cor = sent_tot = sent_cor = amb_tot = amb_cor = 0
rows = []
for bare, acc in test:
    pred = restore_sentence(bare)
    pw, aw = pred.split(), acc.split()
    sent_tot += 1
    sent_cor += (pred == acc)
    for p, a, b in zip(pw, aw, bare.split()):
        tot += 1
        cor += (p == a)
        if b in amb_set:
            amb_tot += 1
            amb_cor += (p == a)
    rows.append({"bare": bare, "gold": acc, "pred": pred})

print(f"word accuracy: {cor}/{tot} = {cor/tot:.1%}")
print(f"sentence accuracy: {sent_cor}/{sent_tot} = {sent_cor/sent_tot:.1%}")
print(f"accuracy tren tu da nghia: {amb_cor}/{amb_tot} = {amb_cor/amb_tot:.1%}")
print("Moc doi chieu (dinh nghia chung): tu dien 77,4% / 2,6% / 74,6% | bigram 91,9% / 30,8% / 91,3% | PhoBERT 83,4% / 6,7% / 81,8%")

with open("/kaggle/working/selfbuilt_test_predictions.csv", "w",
          newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["bare", "gold", "pred"])
    w.writeheader()
    w.writerows(rows)
print("Da luu du doan test -> /kaggle/working/selfbuilt_test_predictions.csv")
for r in rows[:3]:
    print(f"  IN : {r['bare']}\n  OUT: {r['pred']}\n  GOLD: {r['gold']}")
print("Cell 5 OK")

In [ ]:
# Cell 6 — So sánh lại baseline từ điển + phân tích lỗi model tự build
# (chạy trong cùng notebook này để dùng chung dữ liệu và file dự đoán).
import csv, unicodedata
from collections import Counter, defaultdict

# --- 6a. Tái tạo baseline từ điển (dạng có dấu phổ biến nhất) ---
# Đếm tần suất từng dạng có dấu của mỗi từ không dấu trong train.
form_counts = defaultdict(Counter)
for bare, acc in train:
    for b, a in zip(bare.split(), acc.split()):
        form_counts[b][a] += 1
dict_best = {b: c.most_common(1)[0][0] for b, c in form_counts.items()}

# Đo lại 3 metric của từ điển trên test (kiểm chứng với Notebook 1:
# word 77,4% / sentence 2,6%). Accuracy từ đa nghĩa ở đây tính trên
# tập đa nghĩa của train (mẫu số 207.644) nên có thể lệch nhẹ so với
# 74,7% của Notebook 1 (mẫu số 209.279) — đó là định nghĩa chung mới.
d_w = d_w_ok = d_s = d_s_ok = d_a = d_a_ok = 0
for bare, acc in test:
    words, golds = bare.split(), acc.split()
    preds = [dict_best.get(b, b) for b in words]
    ok_sent = True
    for b, g, p in zip(words, golds, preds):
        d_w += 1
        ok = (p == g)
        d_w_ok += ok
        ok_sent = ok_sent and ok
        if b in amb_set:
            d_a += 1
            d_a_ok += ok
    d_s += 1
    d_s_ok += ok_sent
dict_metrics = (d_w_ok/d_w, d_s_ok/d_s, d_a_ok/d_a)
print(f"từ điển (tái tạo) — word: {d_w_ok}/{d_w} = {dict_metrics[0]:.1%} | "
      f"sentence: {d_s_ok}/{d_s} = {dict_metrics[1]:.1%} | "
      f"từ đa nghĩa: {d_a_ok}/{d_a} = {dict_metrics[2]:.1%}")
print("Mốc Notebook 1: word 77,4% | sentence 2,6% | từ đa nghĩa 74,7%")

# --- 6b. Phân tích lỗi của model tự build ---
# Đọc lại file dự đoán đã lưu ở Cell 5 (để cell này chạy độc lập).
rows6 = []
with open("/kaggle/working/selfbuilt_test_predictions.csv",
          encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows6.append(r)

# Dấu thanh trong tiếng Việt (5 dấu): huyền (U+0300), sắc (U+0301),
# hỏi (U+0309), ngã (U+0303), nặng (U+0323). Bỏ các dấu này khỏi từ
# để kiểm tra 2 từ có khác nhau CHỈ ở thanh điệu hay không.
TONE_MARKS = {"\u0300", "\u0301", "\u0309", "\u0303", "\u0323"}

def strip_tones(word):
    return "".join(c for c in unicodedata.normalize("NFD", word)
                   if c not in TONE_MARKS)

train_words = set()
for bare, _ in train:
    train_words.update(bare.split())

err_total = err_tone_only = err_unseen = err_amb = 0
tot_words = 0
confusions = Counter()
err_examples = []
for r in rows6:
    for b, g, p in zip(r["bare"].split(), r["gold"].split(), r["pred"].split()):
        tot_words += 1
        if p == g:
            continue
        err_total += 1
        if strip_tones(p) == strip_tones(g):
            err_tone_only += 1
        if b not in train_words:
            err_unseen += 1
        if b in amb_set:
            err_amb += 1
        confusions[(g, p)] += 1
        if len(err_examples) < 15:
            err_examples.append((r["bare"], r["gold"], r["pred"], g, p))
print(f"tổng số từ sai: {err_total} / {tot_words}")
print(f"sai chỉ ở thanh điệu: {err_tone_only} ({err_tone_only/err_total:.1%} số lỗi)")
print(f"sai ở từ chưa thấy trong train: {err_unseen} ({err_unseen/err_total:.1%})")
print(f"sai ở từ đa nghĩa: {err_amb} ({err_amb/err_total:.1%})")
print("10 cặp nhầm lẫn phổ biến nhất (gold -> pred):")
for (g, p), c in confusions.most_common(10):
    print(f"  {g!r} -> {p!r}: {c} lần")
print("Ví dụ câu có lỗi:")
for bare, gold, pred, g, p in err_examples[:10]:
    print(f"  IN : {bare}")
    print(f"  OUT: {pred}")
    print(f"  GOLD: {gold}   [{g!r} -> {p!r}]")

In [ ]:
# Cell 7 — Tái tạo baseline bigram + Viterbi để có file dự đoán
# (kiểm chứng với Notebook 1: 91,9% / 31,0% / 91,2%).
import csv, math
from collections import Counter

# Đếm unigram/bigram trên câu có dấu trong train.
# Lưu ý: bản này thêm token </s> (Notebook 1 không có) nhưng bước giải mã
# không dùng tới </s> nên không ảnh hưởng kết quả; chênh 0,2pp ở sentence
# accuracy so với Notebook 1 là do cách phá tie khác nhau (ở đây duyệt ứng
# viên theo thứ tự sorted, Notebook 1 duyệt set) — đã ghi trong báo cáo §4.4.
unigram = Counter()
bigram = Counter()
for _, acc in train:
    words = ["<s>"] + acc.split() + ["</s>"]
    unigram.update(words)
    bigram.update(zip(words, words[1:]))
V = len(unigram)
ALPHA = 0.1

def bigram_logprob(prev, w):
    # Làm mịn cộng alpha (giống Notebook 1).
    return math.log((bigram[(prev, w)] + ALPHA) /
                    (unigram[prev] + ALPHA * V))

def viterbi_decode(bare_words):
    # Viterbi trên các dạng có dấu ứng viên của từng từ (từ điển
    # forms của Cell 2); từ chưa thấy trong train thì giữ nguyên.
    cands = [sorted(forms.get(b, {b})) for b in bare_words]
    dp = [{c: bigram_logprob("<s>", c) for c in cands[0]}]
    bp = []
    for i in range(1, len(bare_words)):
        prev_dp = dp[-1]
        cur, cur_bp = {}, {}
        for c in cands[i]:
            best_prev, best_score = None, float("-inf")
            for pc, ps in prev_dp.items():
                s = ps + bigram_logprob(pc, c)
                if s > best_score:
                    best_score, best_prev = s, pc
            cur[c] = best_score
            cur_bp[c] = best_prev
        dp.append(cur)
        bp.append(cur_bp)
    last = max(dp[-1], key=dp[-1].get)
    out = [last]
    for i in range(len(bare_words) - 1, 0, -1):
        last = bp[i - 1][last]
        out.append(last)
    return out[::-1]

# Đo 3 metric của bigram trên test + lưu dự đoán.
b_w = b_w_ok = b_s = b_s_ok = b_a = b_a_ok = 0
bigram_rows = []
for bare, acc in test:
    words, golds = bare.split(), acc.split()
    preds = viterbi_decode(words)
    ok_sent = True
    for b, g, p in zip(words, golds, preds):
        b_w += 1
        ok = (p == g)
        b_w_ok += ok
        ok_sent = ok_sent and ok
        if b in amb_set:
            b_a += 1
            b_a_ok += ok
    b_s += 1
    b_s_ok += ok_sent
    bigram_rows.append({"bare": bare, "gold": acc,
                        "pred": " ".join(preds)})
bigram_metrics = (b_w_ok/b_w, b_s_ok/b_s, b_a_ok/b_a)
print(f"bigram (tái tạo) — word: {b_w_ok}/{b_w} = {bigram_metrics[0]:.1%} | "
      f"sentence: {b_s_ok}/{b_s} = {bigram_metrics[1]:.1%} | "
      f"từ đa nghĩa: {b_a_ok}/{b_a} = {bigram_metrics[2]:.1%}")
print("Mốc Notebook 1: word 91,9% | sentence 31,0% | từ đa nghĩa 91,2%")

with open("/kaggle/working/bigram_test_predictions.csv", "w",
          newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["bare", "gold", "pred"])
    w.writeheader()
    w.writerows(bigram_rows)
print("Da luu du doan bigram -> /kaggle/working/bigram_test_predictions.csv")

In [ ]:
# Cell 8 — Bảng so sánh 4 hệ + biểu đồ.
# Từ điển và bigram: số tái tạo ở Cell 6-7 (cùng định nghĩa tập đa
# nghĩa train). Tự build: tính lại từ file dự đoán. PhoBERT: số đã
# đo ở notebook PhoBERT Version 3 (83,4% / 6,7% / 81,8%).
import csv
import matplotlib.pyplot as plt

# Tính lại 3 metric của tự build từ file dự đoán (kiểm chứng Cell 5).
s_w = s_w_ok = s_s = s_s_ok = s_a = s_a_ok = 0
with open("/kaggle/working/selfbuilt_test_predictions.csv",
          encoding="utf-8") as f:
    for r in csv.DictReader(f):
        words, golds = r["bare"].split(), r["gold"].split()
        preds = r["pred"].split()
        ok_sent = True
        for b, g, p in zip(words, golds, preds):
            s_w += 1
            ok = (p == g)
            s_w_ok += ok
            ok_sent = ok_sent and ok
            if b in amb_set:
                s_a += 1
                s_a_ok += ok
        s_s += 1
        s_s_ok += ok_sent
self_metrics = (s_w_ok/s_w, s_s_ok/s_s, s_a_ok/s_a)
print(f"tự build (tính lại) — word: {self_metrics[0]:.1%} | "
      f"sentence: {self_metrics[1]:.1%} | từ đa nghĩa: {self_metrics[2]:.1%}")

# PhoBERT: tính lại từ file dự đoán nếu có. Tìm file ở /kaggle/working
# (upload tay từ Notebook 2) rồi ở /kaggle/input (dạng Kaggle Dataset).
# Nếu không có file thì dùng số đã đo ở Notebook 2 Version 3 (83,4% /
# 6,7% / 81,8%) và in cảnh báo rõ ràng — lúc đó bảng đang trộn số
# PhoBERT cũ (16/11) với số mới của 3 hệ kia.
import glob as _glob
import os as _os
_phobert_paths = (["/kaggle/working/phobert_predictions.csv"] +
                  sorted(_glob.glob("/kaggle/input/*/phobert_predictions.csv")))
_phobert_file = next((p for p in _phobert_paths if _os.path.exists(p)), None)
if _phobert_file is not None:
    p_w = p_w_ok = p_s = p_s_ok = p_a = p_a_ok = 0
    with open(_phobert_file, encoding="utf-8") as f:
        for r in csv.DictReader(f):
            words, golds = r["bare"].split(), r["gold"].split()
            preds = r["pred"].split()
            ok_sent = True
            for b, g, p in zip(words, golds, preds):
                p_w += 1
                ok = (p == g)
                p_w_ok += ok
                ok_sent = ok_sent and ok
                if b in amb_set:
                    p_a += 1
                    p_a_ok += ok
            p_s += 1
            p_s_ok += ok_sent
    phobert_metrics = (p_w_ok / p_w, p_s_ok / p_s, p_a_ok / p_a)
    print(f"PhoBERT (tinh tu {_phobert_file}) — word: {phobert_metrics[0]:.1%} | "
          f"sentence: {phobert_metrics[1]:.1%} | tu da nghia: {phobert_metrics[2]:.1%}")
else:
    phobert_metrics = (0.834, 0.067, 0.818)
    print("CANH BAO: khong tim thay phobert_predictions.csv — bang dang "
          "tron so PhoBERT cu (83,4% / 6,7% / 81,8%, do 16/11) voi so "
          "moi cua 3 he kia.")
systems = ["Từ điển", "Bigram", "PhoBERT", "Tự build"]
metrics = [dict_metrics, bigram_metrics, phobert_metrics, self_metrics]

print()
print("Hệ thống   | Word  | Sentence | Từ đa nghĩa")
print("-----------|-------|----------|------------")
for s, (w, se, a) in zip(systems, metrics):
    print(f"{s:<10} | {w*100:5.1f}% | {se*100:6.1f}% | {a*100:8.1f}%")

word_acc = [m[0] * 100 for m in metrics]
sent_acc = [m[1] * 100 for m in metrics]
amb_acc = [m[2] * 100 for m in metrics]
x = range(len(systems))
plt.figure(figsize=(8, 4.5))
plt.bar([i - 0.25 for i in x], word_acc, 0.25, label="Word")
plt.bar([i for i in x], sent_acc, 0.25, label="Sentence")
plt.bar([i + 0.25 for i in x], amb_acc, 0.25, label="Từ đa nghĩa")
plt.xticks(list(x), systems)
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.title("So sánh 4 hệ khôi phục dấu (test 10k câu)")
plt.legend()
plt.tight_layout()
plt.savefig("/kaggle/working/comparison_chart.png", dpi=150)
plt.show()
print("Da luu bieu do -> /kaggle/working/comparison_chart.png")

In [ ]:
# Cell 9 — Demo Gradio: gõ câu không dấu, nhận câu có dấu
# (chạy trong notebook này để dùng lại model đã train ở Cell 4).
# Cài gradio (Kaggle chưa có sẵn), nạp lại checkpoint tốt nhất,
# rồi mở demo. Link public (share=True) chỉ sống khi phiên chạy.
!pip install gradio -q

import re
import torch
import gradio as gr

# Nạp lại model tốt nhất (để demo chạy độc lập, không phụ thuộc
# model còn trong bộ nhớ từ Cell 4).
demo_model = SmallTransformer(len(vocab), NUM_LABELS).to(device)
demo_model.load_state_dict(torch.load("/kaggle/working/selfbuilt_best.pt"))
demo_model.eval()
print("da nap checkpoint tot nhat cho demo")

def demo_predict(bare_words):
    # Khôi phục dấu bằng checkpoint tốt nhất (demo_model), dùng chung
    # logic với restore_sentence ở Cell 3.
    return restore_with_model(demo_model, " ".join(bare_words))

def restore_text(text):
    # Tách đoạn văn thành câu (giữ dấu kết thúc câu), khôi phục từng câu.
    sents = re.findall(r"[^.!?]+[.!?]*", text)
    out = []
    for s in sents:
        m = re.match(r"^(.*?)([.!?]*)$", s.strip())
        body, punct = m.group(1), m.group(2)
        out.append(demo_predict(body.split()) + punct)
    return " ".join(out)

demo = gr.Interface(
    fn=restore_text,
    inputs=gr.Textbox(lines=3, label="Câu không dấu",
                      placeholder="Nhập câu tiếng Việt không dấu..."),
    outputs=gr.Textbox(lines=3, label="Câu đã khôi phục dấu"),
    title="Khôi phục dấu tiếng Việt — Transformer tự build",
    description="Model Transformer tự build (word accuracy 92,5% trên test 10k câu).",
    examples=[
        ["xin chao cac ban, toi ten la hao"],
        ["toi dang hoc tieng viet va rat thich mon nay"],
        ["ha noi la thu do cua viet nam"],
    ],
)

# prevent_thread_lock=True để cell kết thúc ngay thay vì treo ở demo
# (cần thiết để Save & Run All không kẹt ở cell này). Nếu môi trường
# commit không tạo được link share thì dùng demo nội bộ.
try:
    demo.launch(share=True, prevent_thread_lock=True)
    print("Demo dang chay — mo link share o tren de thu.")
except Exception as e:
    print("Khong tao duoc link share:", e)
    demo.launch(prevent_thread_lock=True)